In [7]:
import os
import csv
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import shutil
import time

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset
from torch import nn
from warnings import filterwarnings
from xgboost import XGBClassifier

In [8]:
class Encode(nn.Module):
    def __init__(self, in_channels, embeddings):
        super().__init__()

        # Cap hidden sizes to keep memory manageable for very high-dimensional omics data.
        h1 = min(512, max(128, in_channels // 16))
        h2 = min(128, max(32, in_channels // 64))

        self.relu = nn.ReLU()
        self.dense1 = nn.Linear(in_channels, h1)
        self.bn1 = nn.BatchNorm1d(h1)
        self.dense2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2)

        self.mu = nn.Linear(h2, embeddings)
        self.sigma = nn.Linear(h2, embeddings)

    def forward(self, x):
        x = self.bn1(self.relu(self.dense1(x)))
        x = self.bn2(self.relu(self.dense2(x)))
        return self.mu(x), self.sigma(x)


class Decode(nn.Module):
    def __init__(self, embeddings, out_channel):
        super().__init__()

        h1 = min(128, max(32, out_channel // 64))
        h2 = min(512, max(128, out_channel // 16))

        self.relu = nn.ReLU()
        self.dense1 = nn.Linear(embeddings, h1)
        self.bn1 = nn.BatchNorm1d(h1)
        self.dense2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2)
        self.out = nn.Linear(h2, out_channel)

    def forward(self, x):
        x = self.bn1(self.relu(self.dense1(x)))
        x = self.bn2(self.relu(self.dense2(x)))
        return self.out(x)


class ConvVariationalAutoEncoder(nn.Module):
    """Tabular variational autoencoder model."""

    def __init__(self, in_channels, out_channel, embeddings):
        super().__init__()
        self.encoder = Encode(in_channels, embeddings)
        self.decoder = Decode(embeddings, out_channel)

    def reparameterize(self, mu, sigma):
        std = torch.exp(0.5 * sigma)
        epsilon = torch.randn_like(std)
        return epsilon * std + mu

    def forward(self, x):
        mu, sigma = self.encoder(x)
        z = self.reparameterize(mu, sigma)
        x_hat = self.decoder(z)
        return x_hat, mu, sigma

    def encode(self, x):
        mu, _ = self.encoder(x)
        return mu

In [9]:
# Loss is computed inside vae_loss() in the next cell.

In [10]:
# Optimizer is created inside fit_vae() in the next cell.

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
    recon = F.mse_loss(recon_x, x, reduction="mean")
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kld, recon, kld


def fit_vae(model, x_train, epochs=100, batch_size=64, lr=1e-3, beta=0.1, device="cpu"):
    model = model.to(device)
    x_tensor = torch.tensor(x_train, dtype=torch.float32)
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        for (xb,) in loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            recon, mu, logvar = model(xb)
            loss, _, _ = vae_loss(recon, xb, mu, logvar, beta=beta)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{epochs} | loss: {epoch_loss / len(x_train):.6f}")

    return model


@torch.no_grad()
def encode_latent(model, x_np, batch_size=512, device="cpu"):
    model.eval()
    model.to(device)

    x_tensor = torch.tensor(x_np, dtype=torch.float32)
    loader = DataLoader(TensorDataset(x_tensor), batch_size=batch_size, shuffle=False)

    latents = []
    for (xb,) in loader:
        xb = xb.to(device)
        z = model.encode(xb)
        latents.append(z.cpu().numpy())

    return np.vstack(latents)


def train_xgb_on_vae_latent(
    X,
    y,
    latent_dim=32,
    test_size=0.2,
    random_state=42,
    vae_epochs=120,
    vae_batch_size=64,
    vae_lr=1e-3,
    vae_beta=0.1,
    apply_scaler=True,
    device="cpu",
    xgb_params=None,
):
    """Train VAE on X, then fit XGBoost classifier on latent representation."""
    X = np.asarray(X, dtype=np.float32)
    y_arr = np.asarray(y)

    # Replace invalid values to keep scaler/training stable.
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    label_encoder = None
    if y_arr.dtype.kind in {"U", "S", "O"}:
        label_encoder = LabelEncoder()
        y_arr = label_encoder.fit_transform(y_arr)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_arr, test_size=test_size, random_state=random_state, stratify=y_arr
    )

    scaler = None
    if apply_scaler:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train).astype(np.float32)
        X_test = scaler.transform(X_test).astype(np.float32)

    vae = ConvVariationalAutoEncoder(
        in_channels=X_train.shape[1],
        out_channel=X_train.shape[1],
        embeddings=latent_dim,
    )
    vae = fit_vae(
        vae,
        X_train,
        epochs=vae_epochs,
        batch_size=vae_batch_size,
        lr=vae_lr,
        beta=vae_beta,
        device=device,
    )

    Z_train = encode_latent(vae, X_train, device=device)
    Z_test = encode_latent(vae, X_test, device=device)

    n_classes = int(np.unique(y_train).shape[0])
    default_params = {
        "n_estimators": 400,
        "max_depth": 7,
        "learning_rate": 0.1,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "tree_method": "hist",
        "random_state": random_state,
    }

    if n_classes == 2:
        default_params.update({"objective": "binary:logistic", "eval_metric": "logloss"})
    else:
        default_params.update({"objective": "multi:softprob", "eval_metric": "mlogloss"})

    if xgb_params is not None:
        default_params.update(xgb_params)

    clf = XGBClassifier(**default_params)
    clf.fit(Z_train, y_train)

    y_pred = clf.predict(Z_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    weighted_f1 = report["weighted avg"]["f1-score"]

    print("Accuracy:", acc)
    print("Weighted F1:", weighted_f1)
    print(classification_report(y_test, y_pred))

    return {
        "vae": vae,
        "xgb": clf,
        "label_encoder": label_encoder,
        "scaler": scaler,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "Z_train": Z_train,
        "Z_test": Z_test,
        "accuracy": acc,
        "weighted_f1": weighted_f1,
    }



In [14]:
# BRCA aligned 5-CSV loader: 4 omics + 1 label file
base_dir = r"c:\Users\ALIENWARE\OneDrive - Hanoi University of Science and Technology\HUST\ML\Main_Dataset\Classification_datasets\GS-BRCA\Aligned"

cnv_path = os.path.join(base_dir, "BRCA_CNV_aligned.csv")
methy_path = os.path.join(base_dir, "BRCA_Methy_aligned.csv")
mirna_path = os.path.join(base_dir, "BRCA_miRNA_aligned.csv")
mrna_path = os.path.join(base_dir, "BRCA_mRNA_aligned.csv")
label_path = os.path.join(base_dir, "BRCA_label_num.csv")

cnv = pd.read_csv(cnv_path, index_col=0).T
methy = pd.read_csv(methy_path, index_col=0).T
mirna = pd.read_csv(mirna_path, index_col=0).T
mrna = pd.read_csv(mrna_path, index_col=0).T
labels = pd.read_csv(label_path)["Label"].astype(int).values

# Keep common samples in same order across all omics matrices.
common_samples = cnv.index.intersection(methy.index).intersection(mirna.index).intersection(mrna.index)
cnv = cnv.loc[common_samples]
methy = methy.loc[common_samples]
mirna = mirna.loc[common_samples]
mrna = mrna.loc[common_samples]

X_df = pd.concat([cnv, methy, mirna, mrna], axis=1)
X = X_df.values.astype(np.float32)
y = labels[: X.shape[0]]

# Speed up training by keeping highest-variance features.

print("X shape after feature cap:", X.shape)
print("y shape:", y.shape)
print("Class counts:")
print(pd.Series(y).value_counts().sort_index())

artifacts = train_xgb_on_vae_latent(
    X,
    y,
    latent_dim=32,
    vae_epochs=120,
    vae_batch_size=32,
    vae_lr=1e-3,
    vae_beta=0.1,
    apply_scaler=True,
    device="cuda",
)

print("Final test accuracy:", artifacts["accuracy"])

X shape after feature cap: (671, 34045)
y shape: (671,)
Class counts:
0    353
1     42
2    132
3     31
4    113
Name: count, dtype: int64
Epoch   1/120 | loss: 1.168055
Epoch  20/120 | loss: 0.711858
Epoch  40/120 | loss: 0.615082
Epoch  60/120 | loss: 0.527200
Epoch  80/120 | loss: 0.467165
Epoch 100/120 | loss: 0.420476
Epoch 120/120 | loss: 0.375099
Accuracy: 0.7925925925925926
              precision    recall  f1-score   support

           0       0.82      0.85      0.83        71
           1       0.75      0.75      0.75         8
           2       0.64      0.52      0.57        27
           3       0.50      0.67      0.57         6
           4       0.96      1.00      0.98        23

    accuracy                           0.79       135
   macro avg       0.73      0.76      0.74       135
weighted avg       0.79      0.79      0.79       135

Final test accuracy: 0.7925925925925926
